# EXP-2026-002 / Q4-P — `best_epoch = 0` 원인 분리 진단

**상태: EXPLORATORY DIAGNOSTIC / FULL RESULT NOT RUN.**

Q4-O(NO-GO)의 Arm C가 25개 (seed × fold) 전부에서 첫 학습 epoch 완료 후 체크포인트
(`best_epoch = 0`)를 선택한 원인을 분리한다: 즉시 유해(H1) / 첫 epoch 후 과적합(H2) /
LR·alpha overshoot(H3) / selector 불일치(H4).

- spec: `experiments/specs/EXP-2026-002-q4p-best-epoch-zero-diagnostic.md`
- 고정: Q4-O의 데이터·fold map·seeds·모델·offset. 바뀌는 것은 **학습 trajectory와 checkpoint 정의**뿐.
- 학습 전 상태를 `epoch = -1` 체크포인트로 **직접 평가**한다 (Q4-O가 하지 못한 비교).
- 세 schedule(S0/S1/S2) × 세 selector(SEL0/SEL1/SEL2), 고정 24 epoch, patience 없음.
- Q4-O의 측정 artifact는 읽지도, 쓰지도 않는다 — 완전히 별도 run 디렉터리를 만든다.

## 1. Drive mount + 최신 코드

In [ ]:
import os, sys, subprocess, time, importlib

REPO_URL     = "https://github.com/ehdbddl06001-ui/my-github-test.git"
REPO_BRANCH  = "claude/exp-2026-002-q4p-best-epoch-zero-diagnostic"
REPO_DIR     = "/content/my-github-test"
NEED_Q4O     = 4          # Q4-O module version this design depends on
NEED_Q4P     = 1

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as exc:
    print("not Colab:", exc)
    DRIVE_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT", "/content")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard",
                    f"origin/{REPO_BRANCH}"], check=True)

MOD_DIR = os.path.join(REPO_DIR, "mit-bih")
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

# Purge stale imports — a kernel that holds an old module silently ignores git pull.
for _name in [m for m in sys.modules
              if m.startswith(("q4o_leakage_free_residual",
                               "q4p_best_epoch_zero_diagnostic"))]:
    del sys.modules[_name]
importlib.invalidate_caches()

import q4o_leakage_free_residual as Q4O
import q4p_best_epoch_zero_diagnostic as QP

assert Q4O.MODULE_VERSION >= NEED_Q4O, (
    f"stale Q4-O module {Q4O.MODULE_VERSION} < {NEED_Q4O} - restart the runtime")
check = QP.self_check(min_version=NEED_Q4P)
print("q4p module :", check["module_file"])
print("version    :", check["module_version"], "-", check["module_build"])
print("status     :", check["status"])
print("repo commit:", Q4O.git_commit_sha(REPO_DIR))
print("packages   :", Q4O.package_versions())
print("gpu        :", Q4O.gpu_info())

## 2. 테스트 먼저 (Q4-O 스위트 + Q4-P 스위트)

In [ ]:
for test_file in ("test_q4o_leakage_free_residual.py",
                  "test_q4p_best_epoch_zero_diagnostic.py"):
    rc = subprocess.run([sys.executable, os.path.join(MOD_DIR, test_file)],
                        capture_output=True, text=True)
    print(rc.stdout[-1500:])
    if rc.returncode != 0:
        print(rc.stderr[-2500:])
        raise SystemExit(f"{test_file} failed - do not run anything further")
print("both suites passed in a subprocess; in-kernel versions re-verified below")
assert QP.MODULE_VERSION >= NEED_Q4P
QP.self_check(min_version=NEED_Q4P)
print("in-kernel q4p module verified")

## 3. 모드 선택 — 동시에 정확히 하나만

In [ ]:
# ---------------------------------------------------------------------------
# DESIGN_ONLY           : 설계 요약만 출력. 아무것도 학습/실행하지 않는다. (기본)
# SMOKE                 : CPU synthetic smoke run — 배관 검증. 과학적 의미 없음.
# FULL_RUN              : 실제 GPU run (svdb_data5.npz). 이번 작업에서는 실행 금지.
# ANALYZE_EXISTING_RUN  : 완료된 Q4-P run 번들의 보고만 재생성.
# ---------------------------------------------------------------------------
DESIGN_ONLY          = True
SMOKE                = False
FULL_RUN             = False
ANALYZE_EXISTING_RUN = False
EXISTING_RUN_DIR     = ""     # ANALYZE_EXISTING_RUN=True 일 때만 사용

_modes = [DESIGN_ONLY, SMOKE, FULL_RUN, ANALYZE_EXISTING_RUN]
assert sum(bool(m) for m in _modes) == 1, (
    f"exactly ONE mode must be active, got {_modes}")

DATA_PATH = os.path.join(DRIVE_ROOT, "mitbih", "svdb_data5.npz")
PROJECT   = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
RUNS_DIR  = os.path.join(PROJECT, "runs")

MODE = ("DESIGN_ONLY" if DESIGN_ONLY else "SMOKE" if SMOKE
        else "FULL_RUN" if FULL_RUN else "ANALYZE_EXISTING_RUN")
print("MODE :", MODE)
print("schedules :", list(QP.SCHEDULES))
print("selectors :", list(QP.SELECTORS), "(primary:", QP.PRIMARY_SELECTOR + ")")
print("epochs    :", QP.N_EPOCHS, "fixed - patience never stops the optimizer")
print("checkpoint: epoch -1 (pre-training) is a first-class candidate")

## 4. 설계 요약 (DESIGN_ONLY)

In [ ]:
if DESIGN_ONLY:
    n_traj = 2 * len(QP.SCHEDULES) * len(Q4O.TRAIN_SEEDS) * Q4O.N_OUTER_FOLDS
    print("=" * 74)
    print("EXP-2026-002 / Q4-P - DESIGN SUMMARY (nothing executed)")
    print("=" * 74)
    print(f"arms          : C (real waveform) + D (within-record shuffle); A as reference")
    print(f"trajectories  : {n_traj} = 2 arms x {len(QP.SCHEDULES)} schedules x "
          f"{len(Q4O.TRAIN_SEEDS)} seeds x {Q4O.N_OUTER_FOLDS} folds")
    print(f"checkpoints   : {QP.N_EPOCHS + 1} per trajectory (epoch -1 .. {QP.N_EPOCHS - 1})")
    print(f"tie tolerance : BCE {QP.TIE_TOL_BCE} / k-sweep {QP.TIE_TOL_KSW} "
          f"(tie -> earliest checkpoint incl. -1)")
    print(f"grad logging  : first {QP.GRAD_LOG_STEPS} optimizer steps per trajectory")
    print()
    print("pre-registered decision branches:")
    for b in QP.DECISION_BRANCHES:
        print("  -", b)
    print()
    print("FULL RESULT NOT RUN - every result cell below shows NOT RUN.")
else:
    print("skipped - not in DESIGN_ONLY mode")

## 5. 실행 (SMOKE / FULL_RUN / ANALYZE_EXISTING_RUN)

In [ ]:
result = None
OUT_DIR = None
if DESIGN_ONLY:
    print("NOT RUN - DESIGN_ONLY mode performs no execution.")
elif SMOKE:
    OUT_DIR = "/content/q4p_smoke"
    cohort = Q4O.synthetic_cohort(n_record=8, n_beat=110, seed=17, n_unscorable=2)
    prov = {"abs_path": "<synthetic>", "file_name": "<synthetic>",
            "sha256": "<synthetic>", "synthetic": True}
    result = QP.run_diagnostic(cohort, prov, OUT_DIR, seeds=Q4O.TRAIN_SEEDS[:2],
                               epochs=2, batch=256, n_boot=100, device="cpu",
                               smoke=True)
    print("SMOKE ok - no scientific meaning. verdict field exercised:",
          result["decision_tree"]["verdict"])
elif FULL_RUN:
    assert os.path.exists(DATA_PATH), (
        f"{DATA_PATH} not found. Do NOT substitute svdb_data.npz - stop and "
        f"report a blocker.")
    cohort, prov = Q4O.load_cohort(DATA_PATH)
    TIMESTAMP = time.strftime("%Y%m%dT%H%M", time.gmtime())
    OUT_DIR = os.path.join(RUNS_DIR, QP.run_dir_name(TIMESTAMP))
    n_traj = 2 * len(QP.SCHEDULES) * len(Q4O.TRAIN_SEEDS) * Q4O.N_OUTER_FOLDS
    print("data sha256 :", prov["sha256"])
    print("git sha     :", Q4O.git_commit_sha(REPO_DIR))
    print("out dir     :", OUT_DIR)
    print("plan        :", n_traj, "trajectories x", QP.N_EPOCHS,
          "epochs (+ epoch -1 eval each)")
    print("expect      : roughly 6x Q4-O's Arm C wall time on a T4 "
          "(3 schedules x 2 arms, fixed 24 epochs, no early stop)")
    t0 = time.time()
    log = Q4O.RunLog()   # run_diagnostic logs [k/N] progress lines -> live ETA
    result = QP.run_diagnostic(cohort, prov, OUT_DIR, seeds=Q4O.TRAIN_SEEDS,
                               epochs=QP.N_EPOCHS, batch=Q4O.DL_BATCH,
                               n_boot=2000, smoke=False, log=log)
    print(f"wall time: {(time.time() - t0) / 60:.1f} min")
elif ANALYZE_EXISTING_RUN:
    OUT_DIR = (EXISTING_RUN_DIR if os.path.isabs(EXISTING_RUN_DIR)
               else os.path.join(RUNS_DIR, EXISTING_RUN_DIR))
    assert os.path.isdir(OUT_DIR), f"run bundle not found: {OUT_DIR}"
    QP.verify_bundle(OUT_DIR)          # incomplete bundle -> hard fail
    import json as _json
    result = _json.load(open(os.path.join(OUT_DIR, "result.json"),
                             encoding="utf-8"))
    hist = _json.load(open(os.path.join(OUT_DIR, "training_history.json"),
                           encoding="utf-8"))
    cohort, _prov = Q4O.load_cohort(DATA_PATH)
    fm = {int(k): int(v) for k, v in _json.load(open(os.path.join(
        OUT_DIR, "fold_map.json")))["record_to_fold"].items()}
    QP._write_figures_and_report(OUT_DIR, result, hist, cohort, sorted(fm),
                                 result.get("summary", {}).get("seeds",
                                 list(Q4O.TRAIN_SEEDS)), Q4O.RunLog())
    print("re-report complete - measured artifacts fingerprint-verified")

## 6. 원인 판정 (decision tree)

In [ ]:
if result is None:
    print("NOT RUN - no measured decision tree exists yet.")
else:
    dec = result["decision_tree"]
    for b in QP.DECISION_BRANCHES:
        info = dec["branches"].get(b, {})
        state = ("NOT EVALUABLE" if not info.get("evaluable")
                 else ("FIRES" if info.get("fires") else "-"))
        print(f"{state:>14}  {b}")
    print()
    print("verdict:", dec["verdict"])
    if result.get("smoke"):
        print("(SYNTHETIC smoke - the verdict above has no scientific meaning)")

## 7. 핵심 표 — best epoch 분포와 C−D

In [ ]:
if result is None:
    print("NOT RUN")
else:
    s = result["summary"]
    print("P(best = -1 / 0 / >0), primary selector", result["primary_selector"])
    for arm in (Q4O.ARM_C, Q4O.ARM_D):
        for sch in QP.SCHEDULE_NAMES:
            d = s["best_epoch_dist"][arm][sch][result["primary_selector"]]
            print(f"  {arm:28s} {sch:14s} "
                  f"P(-1)={d['p_pretrain']:.2f} P(0)={d['p_epoch0']:.2f} "
                  f"P(>0)={d['p_later']:.2f}")
    print()
    print("C - D (record k-sweep, paired bootstrap), all schedules x selectors:")
    for sch in QP.SCHEDULE_NAMES:
        for sel in QP.SELECTORS:
            c = s["contrasts"][sch][sel]["C_minus_D"]["record_bootstrap"]
            tag = " <- primary" if sel == result["primary_selector"] else ""
            print(f"  {sch:14s} {sel:18s} {c['mean']:+.4f} "
                  f"[{c['ci_low']:+.4f}, {c['ci_high']:+.4f}]{tag}")

## 8. 그림

In [ ]:
if result is None or OUT_DIR is None:
    print("NOT RUN")
else:
    from IPython.display import display, Markdown, Image
    for name in QP.FIGURE_FILES:
        p = os.path.join(OUT_DIR, "figures", name)
        if os.path.exists(p):
            display(Markdown(f"### `{name}`"))
            display(Image(filename=p))
        else:
            print("missing figure:", name)

## 9. report_summary.md

In [ ]:
if result is None or OUT_DIR is None:
    print("NOT RUN")
else:
    from IPython.display import display, Markdown
    display(Markdown(open(os.path.join(OUT_DIR, "figures", "report_summary.md"),
                          encoding="utf-8").read()))

## 10. 번들 검증

In [ ]:
if result is None or OUT_DIR is None:
    print("NOT RUN - nothing was executed in this session.")
    print("This notebook is the pre-registered runner for EXP-2026-002.")
    print("FULL RESULT NOT RUN.")
else:
    for root, dirs, files in os.walk(OUT_DIR):
        for f_ in sorted(files):
            p = os.path.join(root, f_)
            print(f"  {os.path.relpath(p, OUT_DIR):<58} "
                  f"{os.path.getsize(p):>10,d} bytes")
    QP.verify_bundle(OUT_DIR)
    print()
    print("bundle schema verified")